# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Classification. For each page, the model answers yes or no: is this page declining? The output is a probability, so the content team can also sort pages from most to least likely to be declining, which gives them a priority queue.

In [17]:
import pandas as pd
url = "https://raw.githubusercontent.com/iankitthakur/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(df.shape)
print(list(df.columns))

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
for i, c in enumerate(df.columns):
    print(i, c, "|", df[c].dtype)

0 content_id | object
1 client_id | object
2 search_volume | float64
3 competition | float64
4 competition_level | object
5 cpc | float64
6 content_type | object
7 main_intent | object
8 word_count | float64
9 char_count | float64
10 provider_used | object
11 model_used | object
12 impressions_90d | int64
13 clicks_90d | int64
14 pageviews_90d | int64
15 sessions_90d | int64
16 users_90d | int64
17 engaged_sessions_90d | int64
18 ai_sessions_90d | int64
19 scroll_events_90d | int64
20 days_with_impressions | int64
21 days_with_sessions | int64
22 impressions_last_30d | int64
23 clicks_last_30d | int64
24 sessions_last_30d | int64
25 impressions_prev_30d | int64
26 clicks_prev_30d | int64
27 sessions_prev_30d | int64
28 content_age_days | int64
29 age_tier | object
30 age_tier_order | int64
31 days_since_last_update | int64
32 freshness_tier | object
33 word_count_tier | object
34 char_count_tier | object
35 ctr | float64
36 avg_position | float64
37 engagement_rate | float64
38 scroll_

In [19]:
import urllib.request
u = "https://raw.githubusercontent.com/iankitthakur/flyrank-ml-internship/main/scripts/01_prepare_features.py"
lines = urllib.request.urlopen(u).read().decode().splitlines()
for i, line in enumerate(lines):
    if "label" in line.lower():
        print(i + 1, line)

110     df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
131         "declining_rows": int(df["is_declining_label"].sum()),
132         "declining_rate": float(df["is_declining_label"].mean()) if len(df) else 0.0,


In [20]:
print(df["trend_direction"].value_counts())
print(df["trend_pct"].describe())
print(df[["content_id", "impressions_prev_30d", "impressions_last_30d", "trend_direction", "trend_pct"]].head(10))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64
             content_id  impressions_prev_30d  impressions_last_30d  \
0  content_304f48230142                   987                   578   
1  content_a1fb4e703a9e                  5915                  2501   
2  content_9aa793d4d895                  6089                  2382   
3  content_331d6c4de07b                  4206                  3626   
4  content_d99b7a2d90ca                  6452                  4211   
5  content_d4084a4bc775                  1009                   617   
6  content_9a34b442b552                    13                     1   
7  content_a63219c6e95a                   632                   636   
8  content_5e6c160719bc        

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The target is is_declining_label: 1 when trend_direction is "down", else 0. It comes from a defined rule on observed traffic change, not from a human judgement. It is a proxy for "this page needs attention". One caution: the label is built from the last-30-day and previous-30-day impressions, so those columns (and trend_direction and trend_pct) must not be used as inputs, or the model would leak the answer.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Precision@50: of the 50 pages the model ranks highest, how many are really declining. The team can only review a small number of pages, so the top of the list matters most. Because about 54% of pages are declining, random picking would already score near 0.54, so a good result must be clearly above that.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [24]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

cols = ["content_id", "content_type", "word_count", "content_age_days",
        "days_since_last_update", "avg_position", "ctr",
        "trend_direction", "is_declining_label"]
display(df[cols].head(10))

print(df["is_declining_label"].value_counts())
print("declining rate:", round(df["is_declining_label"].mean(), 3))

,content_id,content_type,word_count,content_age_days,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,keyword article,3221.0,187,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,keyword article,2481.0,445,25,20.3,0.05,down,1
2,content_9aa793d4d895,keyword article,3515.0,141,20,36.5,0.09,down,1
3,content_331d6c4de07b,keyword article,NaN,463,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,keyword article,2803.0,263,14,44.0,0.13,down,1
5,content_d4084a4bc775,keyword article,3080.0,147,20,8.5,0.03,down,1
6,content_9a34b442b552,keyword article,3059.0,90,20,7.0,0.00,down,1
7,content_a63219c6e95a,keyword article,NaN,445,22,21.2,0.06,stable,0
8,content_5e6c160719bc,keyword article,3807.0,90,20,46.0,0.09,down,1
9,content_c27558df2b0c,keyword article,NaN,257,104,4.9,0.16,down,1


is_declining_label
1    16262
0    13738
Name: count, dtype: int64
declining rate: 0.542


One row = one content page (identified by content_id). The table has 30,000 rows, one per page, with columns describing the page and its traffic. The last column, is_declining_label, is the target I would predict: 1 if the page's traffic is going down, 0 if not. About 54% of pages are labelled 1.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

A single rule like "traffic fell" only looks at one column. Declining pages differ by age, position, content type, word count and how recently they were updated, and these signals interact. An if-statement can't weigh them all together. The output supports a content action: refresh the highest-ranked pages first. These are observed, directional, decision-support results, not proof of how Google ranks pages.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.